# ViT-Motion on Kaggle — train / evaluate / interpret

**Prerequisites (attach via *Add Input* on the right):**
1. Your **code** as a Kaggle Dataset (the `vit_motion_kaggle.zip` contents — the
   folder that holds `inspect_dataset.py`, `vit_motion/`, `config_kaggle.yaml`).
2. The **data** dataset you built with `kaggle_build_dataset.ipynb`.

**Settings:** GPU = ON (T4 x2 or P100), Internet = ON (first run downloads the
pretrained ViT encoder weights).


In [ ]:
# 1) Stage the code into a writable dir (it writes artifacts next to itself)
import os, sys, glob, shutil, pathlib
CODE_SRC = None
for p in glob.glob("/kaggle/input/**/inspect_dataset.py", recursive=True):
    CODE_SRC = str(pathlib.Path(p).parent)
    break
assert CODE_SRC, "Attach the code dataset first (must contain inspect_dataset.py)."
WORK = "/kaggle/working/vit_motion_project"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("Code staged at:", WORK)

In [ ]:
# 2) Dependencies (Kaggle already has torch/torchvision; ensure timm + opencv)
!pip -q install "timm>=1.0" "opencv-python>=4.9" >/dev/null
import torch, timm
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(), "| timm", timm.__version__)

In [ ]:
# 3) Auto-detect the data root (parent folder of the experiment dirs)
import glob, pathlib
csvs = [c for c in glob.glob("/kaggle/input/**/samples.csv", recursive=True)
        if not c.startswith(CODE_SRC)]
assert csvs, "No samples.csv found — attach the data dataset."
roots = sorted({str(pathlib.Path(c).parent.parent) for c in csvs}, key=len)
DATA_ROOT = roots[0]
print("experiments:", len(csvs))
print("DATA_ROOT :", DATA_ROOT)

In [ ]:
# 4) Build manifest / normalization / splits (writes to artifacts/manifest/)
!python inspect_dataset.py --config config_kaggle.yaml --data-root "{DATA_ROOT}"

In [ ]:
# 5) Smoke test (K=6, M=4 must match config_kaggle.yaml)
!python smoke_test.py --manifest-dir artifacts/manifest --sequence-length 6 --image-update-interval 4

## 6) Train

First full run downloads the pretrained encoder (Internet ON). 100 epochs on the
full set can be long — set `EPOCHS` lower for a first pass, then raise it. Early
stopping (`patience`) still applies. Checkpoints: `best.pt` / `last.pt`.

To **resume** across Kaggle's session limit, re-run with
`--resume artifacts/runs/vit_motion_temporal_cr_v0_2_1/last.pt`.


In [ ]:
# Write a run config with an adjustable epoch count
EPOCHS = 100   # <-- lower (e.g. 15) for a quick first run
import yaml
cfg = yaml.safe_load(open("config_kaggle.yaml"))
cfg["training"]["epochs"] = int(EPOCHS)
yaml.safe_dump(cfg, open("config_run.yaml", "w"), sort_keys=False, allow_unicode=True)
print("epochs =", EPOCHS)

!python train.py --config config_run.yaml

In [ ]:
# 7) Evaluate one experiment (auto-picks a test experiment if EXP is None)
CKPT = "artifacts/runs/vit_motion_temporal_cr_v0_2_1/best.pt"
EXP = None
import pandas as pd
m = pd.read_csv("artifacts/manifest/manifest.csv")
if EXP is None:
    pool = m[m.split == "test"] if (m.split == "test").any() else m
    EXP = sorted(pool.experiment_id.astype(str).unique())[0]
print("evaluating:", EXP)
!python evaluate_experiment.py --config config_run.yaml --checkpoint "{CKPT}" --experiment "{EXP}"

In [ ]:
# 8) Interpretability — Grad-CAM + Integrated Gradients + attention rollout
#     target in {dx, dy, yaw, norm}
!python interpret_experiment.py --config config_run.yaml --checkpoint "{CKPT}" \
    --experiment "{EXP}" --num-samples 8 --target yaw

In [ ]:
# Preview a couple of interpretability figures inline
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"artifacts/interpretability/{EXP}/*.png"))[:2]:
    print(p)
    display(Image(filename=p))

In [ ]:
# 9) (optional) validation video from the evaluation predictions
!python make_validation_video.py --config config_run.yaml \
    --predictions-csv "artifacts/evaluation/{EXP}/predictions.csv" || echo 'skipped'

In [ ]:
# 10) Bundle everything for download from the notebook Output tab
import shutil
shutil.make_archive("/kaggle/working/vit_motion_artifacts", "zip", "artifacts")
print("Download vit_motion_artifacts.zip from the Output tab.")